# Timonel Probe F2 - Adaptador A2 congelado

Prueba si el adaptador de A2 (entrenado en F1: d<T -> terminar)
generaliza a F2 (d<T Y v<=v_max -> terminar).

Cuadrante clave: Q2 (d<T, v>v_max), etiqueta correcta = ir_a.
Si el modelo dice terminar en Q2, ignora la clausula de velocidad.

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

## 2. Cargar modelo + adaptador A2

In [ ]:
from unsloth import FastLanguageModel
import torch, glob, os

max_seq_length = 1024

_ads = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
_ads = [a for a in _ads if "lora-a2" in a.lower() or "banco_A2" in a]
assert _ads, f"No encontre adapter. Encontrados: {glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)}"
ADAPTER_PATH = os.path.dirname(_ads[0])
print(f"ADAPTER_PATH = {ADAPTER_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
print("[OK] Modelo + adaptador cargado")

## 3. Cargar banco probe

In [ ]:
import json
_ps = glob.glob("/kaggle/input/**/probe.jsonl", recursive=True)
assert _ps, "No encontre probe.jsonl"
PROBE_PATH = _ps[0]
print(f"PROBE_PATH = {PROBE_PATH}")
probe = [json.loads(l) for l in open(PROBE_PATH)]
print(f"probe: {len(probe)} contextos")
from collections import Counter
print(Counter(c["cuadrante"] for c in probe))

## 4. Score forzado

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

SYSTEM = ("Sos el timonel de un velero autonomo. Respondes "
          "UNICAMENTE con el JSON pedido, sin texto adicional.")

@torch.no_grad()
def seq_lp(prompt, completion):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    p = tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids
    c = tokenizer(completion, add_special_tokens=False, return_tensors="pt").input_ids
    ids = torch.cat([p, c], dim=1).to(model.device)
    logits = model(ids).logits[0, p.shape[1]-1:-1].float()
    lp = torch.log_softmax(logits, dim=-1)
    return float(lp.gather(1, ids[0, p.shape[1]:].unsqueeze(1)).sum())

def score(ctx):
    lp_t = seq_lp(ctx["prompt"], '{"accion": "terminar"}')
    lp_i = seq_lp(ctx["prompt"], '{"accion": "ir_a"}')
    return lp_t - lp_i

## 5. Evaluacion por cuadrante

In [ ]:
import numpy as np

def wilson(k, n, z=1.96):
    p = k/n; den = 1+z*z/n
    c = (p+z*z/(2*n))/den
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return c-h, c+h

diff = np.array([score(c) for c in probe])
pred = np.array(["terminar" if x > 0 else "ir_a" for x in diff])
real = np.array([c["label"] for c in probe])
cuad = np.array([c["cuadrante"] for c in probe])

print("=== Accuracy por cuadrante ===")
for q in ["Q1", "Q2", "Q3", "Q4"]:
    ix = cuad == q
    k = int((pred[ix] == real[ix]).sum())
    n = int(ix.sum())
    l, h = wilson(k, n)
    pred_term = (pred[ix] == "terminar").sum()
    print(f"  {q}: n={n} acc={k/n:.3f} IC95=[{l:.3f},{h:.3f}]  pred_terminar={pred_term}/{n}")

print()
print("=== Accuracy global ===")
k = int((pred == real).sum())
l, h = wilson(k, len(pred))
print(f"  {k}/{len(pred)} = {k/len(pred):.3f} IC95=[{l:.3f},{h:.3f}]")

print()
print("=== Cuadrante clave Q2 (d<T, v>v_max, label=ir_a) ===")
ix = cuad == "Q2"
k = int((pred[ix] == real[ix]).sum())
print(f"  aciertos: {k}/{ix.sum()} = {k/ix.sum():.3f}")
print(f"  pred_terminar: {(pred[ix]=='terminar').sum()}/{ix.sum()} (deberia ser 0 si lee el contrato)")

## 6. Veredicto (memo 89 seccion 24, etapa 0)

In [ ]:
q2 = cuad == "Q2"
acc_q2 = (pred[q2] == real[q2]).mean()
pred_term_q2 = (pred[q2] == "terminar").mean()

if acc_q2 >= 0.95:
    v = "LEE EL CONTRATO: generaliza a F2 (Q2 correcto >= 0.95)"
elif pred_term_q2 < 0.5:
    v = "IGNORA LA CLAUSULA: predice terminar en Q2 mayoritariamente"
else:
    v = "INCONCLUSO en Q2"

print(f"acc Q2 = {acc_q2:.3f}  |  pred_terminar Q2 = {pred_term_q2:.3f}")
print(f"VEREDICTO PROBE: {v}")